# MNIST con PyTorch. Red Neuronal Feedforward

Implementación de una red neuronal feedforward para clasificación multiclase del dataset MNIST usando PyTorch.

## 1. Carga del dataset MNIST

In [4]:
import torch
from torch.utils.data import random_split
from torchvision import datasets, transforms
import torch.nn as nn
import torch.optim as optim

# Transformación: convierte imágenes PIL a tensores y normaliza píxeles a [0, 1]
transform = transforms.ToTensor()

# MNIST viene con 60,000 para train y 10,000 para test
full_train_dataset = datasets.MNIST(root='../data', train=True,  download=True, transform=transform)
test_dataset       = datasets.MNIST(root='../data', train=False, download=True, transform=transform)

# 10,000 muestras del train para validación
train_dataset, val_dataset = random_split(full_train_dataset, [50_000, 10_000])

print(f"Train:      {len(train_dataset):,} muestras")
print(f"Validación: {len(val_dataset):,} muestras")
print(f"Test:       {len(test_dataset):,} muestras")

Train:      50,000 muestras
Validación: 10,000 muestras
Test:       10,000 muestras


## 2. Definición de la red neuronal

In [3]:
class FeedForwardNet(nn.Module):
    """    
    Como parámetro se recibe una lista con el número de neuronas por capa oculta

    Arquitectura: Entrada (784) → [Linear → ReLU] x n_capas → Output (10)
    """
    def __init__(self, hidden_sizes: list[int]):
        super().__init__()
        
        layer_sizes = [784] + hidden_sizes + [10]
        
        layers = []
        for i in range(len(layer_sizes) - 1):
            # Añadir capas lineales y ReLU
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i + 1])) 
            if i < len(layer_sizes) - 2:   # No ReLU en la última capa
                layers.append(nn.ReLU())
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)   # Aplana 28x28 a 784
        return self.network(x)


# Ver que todo está bien con la arquitectura
model_test = FeedForwardNet(hidden_sizes=[256, 128])
print(model_test)

FeedForwardNet(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)


## 3. Función de pérdida y optimizador

Aquí usamos Cross Entropy como función de pérdida porque es un problema de clasificación multiclase. Para el optimizador, pienso usar solo adam, pero igual hice una función para comparar con SGD. 

In [7]:
# Función de pérdida
criterion = nn.CrossEntropyLoss()

# Optimizadores 
def make_optimizer(model, optimizer_name='adam', lr=1e-3):
    if optimizer_name == 'adam':
        return optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name == 'sgd':
        return optim.SGD(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Optimizador desconocido: {optimizer_name}")

## 4. Loop de entrenamiento

In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm

def train(model, optimizer, epochs, batch_size=128):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
    
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(1, epochs + 1):
        # Fase de entrenamiento 
        model.train()
        running_loss = 0.0
        
        loop = tqdm(train_loader, desc=f"Época {epoch}/{epochs}", leave=False)
        for X_batch, y_batch in loop:
            optimizer.zero_grad()          # Limpia gradientes
            logits = model(X_batch)        # Forward pass
            loss = criterion(logits, y_batch)
            loss.backward()                # Back propagation
            optimizer.step()               # Actualización de pesos
            
            running_loss += loss.item()
            loop.set_postfix(loss=f"{loss.item():.4f}")
        
        train_loss = running_loss / len(train_loader)
        
        # Fase de validación
        model.eval()
        val_loss = 0.0
        correct  = 0
        
        with torch.no_grad():              # Quitar gradientes para validación
            for X_batch, y_batch in val_loader:
                logits = model(X_batch)
                val_loss += criterion(logits, y_batch).item()
                preds = logits.argmax(dim=1)
                correct += (preds == y_batch).sum().item()
        
        val_loss /= len(val_loader)
        val_acc   = correct / len(val_dataset)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Época {epoch:>2}/{epochs} | "
              f"Train loss: {train_loss:.4f} | "
              f"Val loss: {val_loss:.4f} | "
              f"Val acc: {val_acc:.4f}")
    
    # Retorna un historial con las métricas por época para analizar después.
    return history